# Weather Prediction Pipeline - Exploration

This notebook provides visualization and exploration of the weather prediction pipeline.
All core logic lives in `src/weather_pipeline/` - this notebook imports from there.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from weather_pipeline.session import create_spark_session
from weather_pipeline.config import PipelineConfig

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Results

Load the metrics from the pipeline run.

In [ ]:
# Load results from pipeline run
with open('../results/metrics.json') as f:
    results = json.load(f)

print("Best Hyperparameters:")
for k, v in results['best_params'].items():
    print(f"  {k}: {v}")

## Model Comparison

Compare the GBT model against baselines.

In [ ]:
# Extract test metrics for all models
comparison_data = []
for model_name, metrics_dict in results['models'].items():
    test_metrics = metrics_dict['test']
    comparison_data.append({
        'Model': model_name.replace('_', ' '),
        'MAE': test_metrics['mae'],
        'RMSE': test_metrics['rmse'],
        'R²': test_metrics['r2']
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nModel Comparison (Test Set):")
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize MAE comparison
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(comparison_df['Model'], comparison_df['MAE'], color=['#2ecc71', '#95a5a6', '#95a5a6', '#95a5a6'])
ax.set_ylabel('Mean Absolute Error (°C)')
ax.set_title('Model Comparison - Test Set MAE')
ax.set_ylim(0, max(comparison_df['MAE']) * 1.2)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}°C',
            ha='center', va='bottom')

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## Feature Importance

Visualize which features contribute most to predictions.

In [ ]:
# Top 10 features
importance_df = pd.DataFrame(results['feature_importance'][:10])

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Top 10 Feature Importances')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Data Exploration (Optional)

If you want to explore the raw data, import the pipeline modules:

In [ ]:
# Example: Load and explore data using pipeline modules
# from weather_pipeline.config import DataConfig
# from weather_pipeline.ingest import ingest_data

# spark = create_spark_session()
# config = DataConfig(
#     raw_weather_path='../data/daily_weather.parquet',
#     cities_path='../data/cities.csv',
#     countries_path='../data/countries.csv'
# )
# data = ingest_data(spark, config)
# data.show(5)